# 01 — Data Preparation: Airbnb NYC

Notebook ini mengenali data mentah, membersihkannya, menangani nilai tidak valid, mengecek outlier, lalu membuat kolom kategori baru. Hasilnya disimpan sebagai `airbnb_cleaned_for_dashboard.csv`, yang dipakai oleh `02_airbnb_eda.ipynb` dan dashboard Looker Studio.

- **Input:** `Airbnb_Open_Data.csv` (102.599 baris, 26 kolom; unduh dari Kaggle dan simpan satu folder dengan notebook ini)
- **Output:** `airbnb_cleaned_for_dashboard.csv` (102.045 baris), `docs/validity_check.csv`, `docs/outlier_check.csv`

**Cara pakai:** jalankan **Restart and Run All**. Sel `cek(...)` mencetak ✅ bila hasilnya sama dengan angka yang sudah diverifikasi dan ❌ bila berbeda.

**Alur:**
1. Muat data
2. Gambaran umum data mentah
3. Pembersihan struktur dan teks
4. Penanganan nilai kosong
5. Tipe data
6. Pengecekan dan penanganan nilai tidak valid
7. Pengecekan outlier
8. Kolom kategori baru
9. Penyimpanan

## 1. Memuat data

In [23]:
import os

import numpy as np
import pandas as pd

RAW_FILE = "Airbnb_Open_Data.csv"
OUT_FILE = "airbnb_cleaned_for_dashboard.csv"
DOCS_DIR = "docs"
os.makedirs(DOCS_DIR, exist_ok=True)


def cek(nama, hasil, harapan):
    """Cetak ✅ bila hasil sama dengan angka yang sudah diverifikasi, ❌ bila berbeda."""
    status = "✅" if hasil == harapan else "❌"
    print(f"{status} {nama}: {hasil} (harapan: {harapan})")


df = pd.read_csv(RAW_FILE, low_memory=False)
cek("baris mentah", len(df), 102599)
cek("kolom mentah", df.shape[1], 26)

✅ baris mentah: 102599 (harapan: 102599)
✅ kolom mentah: 26 (harapan: 26)


## 2. Gambaran umum data mentah

In [24]:
df.head()

,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country,...,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules,license
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,$193,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,NaN
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,$28,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,NaN
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,NaN,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,$124,3.0,0.0,NaN,NaN,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",NaN
3,1002755,NaN,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,$74,30.0,270.0,7/5/2019,4.64,4.0,1.0,322.0,NaN,NaN
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,$41,10.0,9.0,11/19/2018,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",NaN


In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 102599 entries, 0 to 102598
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              102599 non-null  int64  
 1   NAME                            102349 non-null  str    
 2   host id                         102599 non-null  int64  
 3   host_identity_verified          102310 non-null  str    
 4   host name                       102193 non-null  str    
 5   neighbourhood group             102570 non-null  str    
 6   neighbourhood                   102583 non-null  str    
 7   lat                             102591 non-null  float64
 8   long                            102591 non-null  float64
 9   country                         102067 non-null  str    
 10  country code                    102468 non-null  str    
 11  instant_bookable                102494 non-null  object 
 12  cancellation_policy        

In [26]:
# kolom wilayah: apakah ada penulisan yang tidak konsisten?
print(df["neighbourhood group"].unique())

<StringArray>
[     'Brooklyn',     'Manhattan',       'brookln',      'manhatan',
        'Queens',             nan, 'Staten Island',         'Bronx']
Length: 8, dtype: str


### Nilai kosong

In [27]:
missing = pd.DataFrame({
    "missing value": df.isna().sum(),
    "percentage(%)": (df.isna().mean() * 100).round(2),
    "data type": df.dtypes.astype(str),
})
missing = missing[missing["missing value"] > 0].sort_values("missing value", ascending=False)
print(missing)

total_missing = int(df.isna().sum().sum())
print(f"\nJumlah missing value: {total_missing} ({total_missing / df.size * 100:.2f}% dari seluruh sel)")

                                missing value  percentage(%) data type
license                                102597         100.00       str
house_rules                             52131          50.81       str
last review                             15893          15.49       str
reviews per month                       15879          15.48   float64
country                                   532           0.52       str
availability 365                          448           0.44   float64
minimum nights                            409           0.40   float64
host name                                 406           0.40       str
review rate number                        326           0.32   float64
calculated host listings count            319           0.31   float64
host_identity_verified                    289           0.28       str
service fee                               273           0.27       str
NAME                                      250           0.24       str
price 

Kolom `license` kosong hampir seluruhnya dan `house_rules` kosong sekitar separuh, jadi keduanya perlu perlakuan khusus.

### Duplikat

In [28]:
n_dup = int(df.duplicated().sum())
cek("baris duplikat", n_dup, 541)
print(f"{n_dup / len(df) * 100:.2f}% dari seluruh baris")

✅ baris duplikat: 541 (harapan: 541)
0.53% dari seluruh baris


In [29]:
df.describe()

,id,host id,lat,long,Construction year,minimum nights,number of reviews,reviews per month,review rate number,calculated host listings count,availability 365
count,1.025990e+05,1.025990e+05,102591.000000,102591.000000,102385.000000,102190.000000,102416.000000,86720.000000,102273.000000,102280.000000,102151.000000
mean,2.914623e+07,4.925411e+10,40.728094,-73.949644,2012.487464,8.135845,27.483743,1.374022,3.279106,7.936605,141.133254
std,1.625751e+07,2.853900e+10,0.055857,0.049521,5.765556,30.553781,49.508954,1.746621,1.284657,32.218780,135.435024
min,1.001254e+06,1.236005e+08,40.499790,-74.249840,2003.000000,-1223.000000,0.000000,0.010000,1.000000,1.000000,-10.000000
25%,1.508581e+07,2.458333e+10,40.688740,-73.982580,2007.000000,2.000000,1.000000,0.220000,2.000000,1.000000,3.000000
50%,2.913660e+07,4.911774e+10,40.722290,-73.954440,2012.000000,3.000000,7.000000,0.740000,3.000000,1.000000,96.000000
75%,4.320120e+07,7.399650e+10,40.762760,-73.932350,2017.000000,5.000000,30.000000,2.000000,4.000000,2.000000,269.000000
max,5.736742e+07,9.876313e+10,40.916970,-73.705220,2022.000000,5645.000000,1024.000000,90.000000,5.000000,332.000000,3677.000000


## 3. Pembersihan struktur dan teks

Nama kolom diseragamkan (huruf kecil, spasi menjadi garis bawah), duplikat dibuang, kolom `license` dibuang karena hampir kosong seluruhnya, dan salah ketik pada nama wilayah dikoreksi.

In [30]:
df.columns = df.columns.str.lower().str.replace(" ", "_")
print(df.columns.tolist())

['id', 'name', 'host_id', 'host_identity_verified', 'host_name', 'neighbourhood_group', 'neighbourhood', 'lat', 'long', 'country', 'country_code', 'instant_bookable', 'cancellation_policy', 'room_type', 'construction_year', 'price', 'service_fee', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'review_rate_number', 'calculated_host_listings_count', 'availability_365', 'house_rules', 'license']


In [31]:
df = df.drop_duplicates()
cek("baris setelah duplikat dibuang", len(df), 102058)

# license kosong 99,9% sehingga tidak informatif
df = df.drop(columns=["license"])

✅ baris setelah duplikat dibuang: 102058 (harapan: 102058)


In [32]:
df["neighbourhood_group"] = df["neighbourhood_group"].replace({
    "brookln": "Brooklyn",
    "manhatan": "Manhattan",
})
print(df["neighbourhood_group"].unique())

<StringArray>
['Brooklyn', 'Manhattan', 'Queens', nan, 'Staten Island', 'Bronx']
Length: 6, dtype: str


### Tipe data teks yang seharusnya angka dan tanggal

`price` dan `service_fee` tertulis dengan simbol `$` dan koma, jadi diubah menjadi angka desimal. `last_review` diubah menjadi tanggal.

In [33]:
for col in ["price", "service_fee"]:
    df[col] = (df[col].astype(str)
               .str.replace("$", "", regex=False)
               .str.replace(",", "", regex=False)
               .astype(float))

df["last_review"] = pd.to_datetime(df["last_review"])

## 4. Penanganan nilai kosong

Aturannya:
- **Penanda sebelum diisi:** `has_house_rules` dibuat dari `house_rules` **sebelum** kolom itu diisi teks pengganti. Kalau dibuat sesudahnya, semua baris pasti terhitung "Yes".
- **Isi teks pengganti** untuk kolom kategori.
- **Isi angka yang logis:** tidak ada ulasan berarti 0 ulasan, dan host yang terdaftar pasti punya minimal 1 listing.
- **Isi median** untuk kolom angka lainnya.
- **`last_review`, `lat`, `long` dibiarkan kosong** supaya tanggal ulasan dan titik peta tidak dimanipulasi.

In [34]:
# harus dibuat sebelum house_rules diisi
df["has_house_rules"] = np.where(df["house_rules"].notna(), "Yes", "No")
print(df["has_house_rules"].value_counts())

has_house_rules
No     51842
Yes    50216
Name: count, dtype: int64


In [35]:
# A. Teks pengganti
df["house_rules"] = df["house_rules"].fillna("No Special Rules")
df["host_identity_verified"] = df["host_identity_verified"].fillna("unconfirmed")
df["name"] = df["name"].fillna("Anonymous")
df["host_name"] = df["host_name"].fillna("Anonymous")
df["country"] = df["country"].fillna("United States")
df["country_code"] = df["country_code"].fillna("US")
df["instant_bookable"] = df["instant_bookable"].fillna("Not Specified")
df["cancellation_policy"] = df["cancellation_policy"].fillna("Not Specified")
df["neighbourhood_group"] = df["neighbourhood_group"].fillna("Unknown")
df["neighbourhood"] = df["neighbourhood"].fillna("Unknown")

# B. Nilai yang logis
df["reviews_per_month"] = df["reviews_per_month"].fillna(0)
df["number_of_reviews"] = df["number_of_reviews"].fillna(0)
df["calculated_host_listings_count"] = df["calculated_host_listings_count"].fillna(1)

# C. Median
kolom_median = ["price", "service_fee", "minimum_nights", "availability_365",
                "construction_year", "review_rate_number"]
for col in kolom_median:
    df[col] = df[col].fillna(df[col].median())

In [36]:
sisa = df.isna().sum()
sisa = sisa[sisa > 0]
print("Sisa nilai kosong (sengaja dibiarkan):")
print(pd.DataFrame({"jumlah kosong": sisa, "persentase (%)": (sisa / len(df) * 100).round(2)}))
cek("kolom yang masih kosong", sorted(sisa.index.tolist()), ["last_review", "lat", "long"])

Sisa nilai kosong (sengaja dibiarkan):
             jumlah kosong  persentase (%)
lat                      8            0.01
long                     8            0.01
last_review          15832           15.51
✅ kolom yang masih kosong: ['last_review', 'lat', 'long'] (harapan: ['last_review', 'lat', 'long'])


## 5. Tipe data

Lima kolom yang berisi bilangan bulat diubah dari desimal ke `int64`. Ini hanya bisa dilakukan setelah semua nilai kosongnya terisi di bagian 4.

In [37]:
kolom_int = ["minimum_nights", "number_of_reviews", "review_rate_number",
             "calculated_host_listings_count", "availability_365"]
df[kolom_int] = df[kolom_int].astype("int64")
print(df[kolom_int].dtypes)

minimum_nights                    int64
number_of_reviews                 int64
review_rate_number                int64
calculated_host_listings_count    int64
availability_365                  int64
dtype: object


## 6. Pengecekan dan penanganan nilai tidak valid

Tujuh aturan bisnis diperiksa. Penanganannya strategi campuran: nilai yang bisa diselamatkan dibatasi (*capping*), baris yang logikanya cacat dibuang.

In [38]:
tahun_ini = pd.Timestamp.today().year

aturan = [
    ("price > 0", df["price"] <= 0),
    ("minimum_nights minimal 1", df["minimum_nights"] <= 0),
    ("availability_365 antara 0 dan 365", (df["availability_365"] < 0) | (df["availability_365"] > 365)),
    ("service_fee >= 0 dan lebih kecil dari price", (df["service_fee"] < 0) | (df["service_fee"] >= df["price"])),
    ("review_rate_number antara 1 dan 5", (df["review_rate_number"] < 1) | (df["review_rate_number"] > 5)),
    ("construction_year tidak di masa depan", df["construction_year"] > tahun_ini),
    ("koordinat bukan 0,0", (df["lat"] == 0) | (df["long"] == 0)),
]

validity = pd.DataFrame(
    [(nama, int(bad.sum()), round(bad.mean() * 100, 2), "✅" if bad.sum() == 0 else "❌")
     for nama, bad in aturan],
    columns=["Aturan validitas", "n tidak valid", "% tidak valid", "Status"],
)
validity

,Aturan validitas,n tidak valid,% tidak valid,Status
0,price > 0,0,0.00,✅
1,minimum_nights minimal 1,13,0.01,❌
2,availability_365 antara 0 dan 365,3185,3.12,❌
3,service_fee >= 0 dan lebih kecil dari price,13,0.01,❌
4,review_rate_number antara 1 dan 5,0,0.00,✅
5,construction_year tidak di masa depan,0,0.00,✅
6,"koordinat bukan 0,0",0,0.00,✅


In [39]:
cek("minimum_nights tidak valid", int(validity.iloc[1]["n tidak valid"]), 13)
cek("availability_365 tidak valid", int(validity.iloc[2]["n tidak valid"]), 3185)
cek("service_fee tidak valid", int(validity.iloc[3]["n tidak valid"]), 13)

✅ minimum_nights tidak valid: 13 (harapan: 13)
✅ availability_365 tidak valid: 3185 (harapan: 3185)
✅ service_fee tidak valid: 13 (harapan: 13)


In [40]:
# 1. Capping: nilai dibatasi ke rentang yang masuk akal
df["availability_365"] = df["availability_365"].clip(lower=0, upper=365)
df["minimum_nights"] = df["minimum_nights"].clip(lower=1)

# 2. Drop: baris dengan service_fee minus atau tidak lebih kecil dari harga sewa
df = df[(df["service_fee"] >= 0) & (df["service_fee"] < df["price"])]

cek("baris setelah penanganan", len(df), 102045)

✅ baris setelah penanganan: 102045 (harapan: 102045)


## 7. Pengecekan outlier

Outlier dideteksi dengan IQR (batas 1,5 kali jangkauan antarkuartil). **Semua outlier dipertahankan.** Nilai ekstrem pada `minimum_nights` (sekitar 18%) dan `calculated_host_listings_count` (sekitar 17%) adalah variasi model bisnis yang valid (misalnya sewa bulanan dan host berskala besar), bukan kesalahan input, dan dapat menjadi insight.

In [41]:
num_cols = ["construction_year", "price", "service_fee", "minimum_nights",
            "number_of_reviews", "reviews_per_month", "review_rate_number",
            "calculated_host_listings_count", "availability_365"]

hasil = {}
for col in num_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((df[col] < lo) | (df[col] > hi)).sum())
    hasil[col] = {"lower_bound": lo, "upper_bound": hi,
                  "n_outlier": n_out, "percentage": n_out / len(df) * 100}

outlier = pd.DataFrame(hasil).round(2)
outlier

,construction_year,price,service_fee,minimum_nights,number_of_reviews,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365
lower_bound,1994.5,-515.5,-103.0,-2.50,-42.50,-2.34,-1.0,-0.50,-394.5
upper_bound,2030.5,1768.5,353.0,9.50,73.50,4.14,7.0,3.50,665.5
n_outlier,0.0,0.0,0.0,18236.00,11695.00,5726.00,0.0,17699.00,0.0
percentage,0.0,0.0,0.0,17.87,11.46,5.61,0.0,17.34,0.0


## 8. Kolom kategori baru

Lima kolom turunan dibuat untuk memudahkan analisis dan dashboard:

| Kolom | Dasar | Kategori |
|---|---|---|
| `price_category` | harga per malam | Budget (sampai 200), Standard (201-500), Business-Class (501-1000), High-End (di atas 1000) |
| `host_type` | jumlah listing per host | Single Host (1), Dual Host (2), Multi Host (3-10), Enterprise (lebih dari 10) |
| `stay_length_category` | minimal menginap | Short Stay (1-3 malam), Weekly Stay (4-7), Monthly Stay (8-30), Yearly-Stay (31-365), Long-Term (lebih dari 365) |
| `review_activity` | jumlah review | No Review (0), Very Low (1-5), Low (6-20), Medium (21-50), High (lebih dari 50) |
| `has_house_rules` | ada tidaknya aturan rumah | Yes, No (sudah dibuat di bagian 4) |

In [42]:
df["price_category"] = pd.cut(
    df["price"], bins=[-np.inf, 200, 500, 1000, np.inf],
    labels=["Budget", "Standard", "Business-Class", "High-End"])

df["host_type"] = pd.cut(
    df["calculated_host_listings_count"], bins=[0, 1, 2, 10, np.inf],
    labels=["Single Host", "Dual Host", "Multi Host", "Enterprise"])

df["stay_length_category"] = pd.cut(
    df["minimum_nights"], bins=[0, 3, 7, 30, 365, np.inf],
    labels=["Short Stay", "Weekly Stay", "Monthly Stay", "Yearly-Stay", "Long-Term"])

df["review_activity"] = pd.cut(
    df["number_of_reviews"], bins=[-1, 0, 5, 20, 50, np.inf],
    labels=["No Review", "Very Low", "Low", "Medium", "High"])

df[["price_category", "host_type", "stay_length_category", "review_activity", "has_house_rules"]].head()

,price_category,host_type,stay_length_category,review_activity,has_house_rules
0,Business-Class,Multi Host,Monthly Stay,Low,Yes
1,Budget,Dual Host,Monthly Stay,Medium,Yes
2,Business-Class,Single Host,Short Stay,No Review,Yes
3,Standard,Single Host,Monthly Stay,High,No
4,Standard,Single Host,Monthly Stay,Low,Yes


In [43]:
for col in ["price_category", "host_type", "stay_length_category", "review_activity", "has_house_rules"]:
    print(df[col].value_counts(dropna=False).to_string(), "\n")

price_category
Business-Class    44555
Standard          26575
High-End          17709
Budget            13206 

host_type
Single Host    63431
Multi Host     16503
Dual Host      14380
Enterprise      7731 

stay_length_category
Short Stay      65228
Weekly Stay     18180
Monthly Stay    16950
Yearly-Stay      1652
Long-Term          35 

review_activity
Very Low     30344
Low          23572
High         17468
No Review    15852
Medium       14809 

has_house_rules
No     51836
Yes    50209 



## 9. Penyimpanan

In [44]:
# has_house_rules dibuat lebih awal; pindahkan ke akhir supaya kolom baru berkumpul di ujung
df = df[[c for c in df.columns if c != "has_house_rules"] + ["has_house_rules"]]

df.to_csv(OUT_FILE, index=False)
validity.to_csv(os.path.join(DOCS_DIR, "validity_check.csv"), index=False)
outlier.to_csv(os.path.join(DOCS_DIR, "outlier_check.csv"))

hasil_csv = pd.read_csv(OUT_FILE, low_memory=False)
cek("shape airbnb_cleaned_for_dashboard.csv", hasil_csv.shape, (102045, 30))
df.info()

✅ shape airbnb_cleaned_for_dashboard.csv: (102045, 30) (harapan: (102045, 30))
<class 'pandas.DataFrame'>
Index: 102045 entries, 0 to 102057
Data columns (total 30 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   id                              102045 non-null  int64         
 1   name                            102045 non-null  str           
 2   host_id                         102045 non-null  int64         
 3   host_identity_verified          102045 non-null  str           
 4   host_name                       102045 non-null  str           
 5   neighbourhood_group             102045 non-null  str           
 6   neighbourhood                   102045 non-null  str           
 7   lat                             102037 non-null  float64       
 8   long                            102037 non-null  float64       
 9   country                         102045 non-null  str          